# PySpark drills 9: Python UDFs and pandas

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-09-python-udfs-and-pandas.ipynb)

Practice for [PySpark 9: Python UDFs and pandas](pyspark-09-python-udfs-and-pandas.ipynb). Use pandas and Python only where the exercise asks for it. The `orders` table is created above.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import pandas as pd

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Replace a UDF with built-ins

**Difficulty:** Easy · about 10 min

A teammate wrote a Python UDF that returns the domain of an email in lower case, or null when there is no `@`. Write
the same logic with built-in functions only. Return `email` and `domain`, ordered by `email` with nulls last.

**Sample input**

**`emails`**

| email |
| --- |
| Ada@Example.COM |
| no-at-sign |
| null |

**Expected output**

| email | domain |
| --- | --- |
| Ada@Example.COM | example.com |
| no-at-sign | null |
| null | null |

In [ ]:
# Sample data for this exercise.
emails = spark.createDataFrame([("Ada@Example.COM",), ("no-at-sign",), (None,)], "email string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = emails.select(
    "email",
    F.when(F.col("email").contains("@"), F.lower(F.element_at(F.split("email", "@"), -1))).alias("domain"),
).orderBy(F.col("email").asc_nulls_last())
result.show()

## Exercise 2: A vectorized pandas UDF

**Difficulty:** Medium · about 10 min

Write a pandas UDF `discounted(amount)` that returns `amount * 0.9` rounded to 2 decimals, as a `double`. Return
`order_id`, `amount` and `discounted` for orders 1 to 4, ordered by `order_id`.

**Expected output**

| order_id | amount | discounted |
| --- | --- | --- |
| 1 | 120.00 | 108.0 |
| 2 | 84.50 | 76.05 |
| 3 | 30.00 | 27.0 |
| 4 | 260.00 | 234.0 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
def discount_batch(amount: pd.Series) -> pd.Series:
    return (amount.astype("float64") * 0.9).round(2)


discounted = F.pandas_udf(discount_batch, "double")  # the same as decorating with @F.pandas_udf("double")
result = orders.where("order_id <= 4").select("order_id", "amount", discounted("amount").alias("discounted")).orderBy("order_id")
result.show()

## Exercise 3: Per-customer logic with applyInPandas

**Difficulty:** Medium · about 15 min

With `groupBy("customer_id").applyInPandas`, return for each customer `customer_id`, `orders` and `largest_share`:
the largest order's share of the customer's total, rounded to 2 decimals. Order by `customer_id`.

**Expected output**

| customer_id | orders | largest_share |
| --- | --- | --- |
| 1 | 3 | 0.51 |
| 2 | 2 | 0.82 |
| 3 | 2 | 0.68 |
| 4 | 2 | 0.93 |
| 5 | 1 | 1.0 |
| 6 | 2 | 0.69 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
def summarize(pdf: pd.DataFrame) -> pd.DataFrame:
    amounts = pdf["amount"].astype("float64")
    return pd.DataFrame({
        "customer_id": [int(pdf["customer_id"].iloc[0])],
        "orders": [len(pdf)],
        "largest_share": [round(amounts.max() / amounts.sum(), 2)],
    })


result = orders.groupBy("customer_id").applyInPandas(
    summarize, schema="customer_id int, orders long, largest_share double"
).orderBy("customer_id")
result.show()

## Exercise 4: Expand rows with mapInPandas

**Difficulty:** Medium · about 15 min

With `mapInPandas`, turn each shipped order into one row per tracking step: steps `packed`, `in_transit` and
`delivered`. Other orders produce no rows. Return `order_id` and `step`, ordered by `order_id` then `step`.

**Expected output**

| order_id | step |
| --- | --- |
| 2 | delivered |
| 2 | in_transit |
| 2 | packed |
| 6 | delivered |
| 6 | in_transit |
| 6 | packed |
| 9 | delivered |
| 9 | in_transit |
| 9 | packed |
| 12 | delivered |
| 12 | in_transit |
| 12 | packed |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
def tracking_steps(batches):
    for pdf in batches:
        shipped = pdf.loc[pdf["status"] == "shipped", ["order_id"]]
        steps = pd.DataFrame({"step": ["packed", "in_transit", "delivered"]})
        yield shipped.merge(steps, how="cross")


result = orders.mapInPandas(tracking_steps, schema="order_id int, step string").orderBy("order_id", "step")
result.show(12)

## Exercise 5: Split codes with a Python UDTF

**Difficulty:** Hard · about 15 min

Write a Python UDTF that takes a string like `"A1|B2"` and yields one row per code with `code` and `position`
(starting at 0), skipping empty codes. Call it on `"A1||B2|C3"` and order by `position`.

**Expected output**

| code | position |
| --- | --- |
| A1 | 0 |
| B2 | 1 |
| C3 | 2 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
class SplitCodesFunction:
    def eval(self, text: str):
        codes = [code for code in (text or "").split("|") if code]
        for position, code in enumerate(codes):
            yield code, position


SplitCodes = F.udtf(SplitCodesFunction, returnType="code string, position int")  # same as the @F.udtf decorator
result = SplitCodes(F.lit("A1||B2|C3")).orderBy("position")
result.show()

**Next:** [PySpark 10: batch pipelines](pyspark-10-batch-pipelines.ipynb)